In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sqlalchemy import create_engine
import os

# Connection

engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={
        "host":     "localhost",
        "port":     5432,
        "dbname":   "IBM",
        "user":     "ibm_user",
        "password": "Aquadevida@1"
    }
)

# Paths

base_path    = "/Users/akshayraghunath/Documents/ibmhr/"
exports_path = base_path + "outputs/exports/"
os.makedirs(exports_path, exist_ok=True)

print("Setup complete!")

Setup complete!


In [4]:
df = pd.read_sql('SELECT * FROM employees', engine)
print(f" Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

 Loaded: 1,544 rows, 32 columns


In [8]:
# Create dummy model

df_model = df.copy()

# Convert target variable  1 = left, 0 = stayed

df_model['Attrition'] = df_model['Attrition'].map({'Yes': 1, 'No': 0})

# Convert OverTime  1 = Yes, 0 = No

df_model['OverTime'] = df_model['OverTime'].map({'Yes': 1, 'No': 0})

# Convert categorical columns dummy variables

df_model = pd.get_dummies(df_model, columns=[
    'Department',
    'JobRole',
    'MaritalStatus',
    'BusinessTravel',
    'EducationField',
    'Gender'
], drop_first=True)

#Drop columns not useful for modelling

df_model = df_model.drop(columns=['HireDate'])

print(f" Features ready!")
print(f"   Columns after encoding: {df_model.shape[1]}")
print(f"   Attrition distribution:")
print(df_model['Attrition'].value_counts())

 Features ready!
   Columns after encoding: 47
   Attrition distribution:
0    1226
1     318
Name: Attrition, dtype: int64


In [10]:
# Correlation
# Select numeric columns only

numeric_cols = df_model.select_dtypes(include=[np.number]).columns.tolist()

# Calculate correlation with Attrition

correlation = df_model[numeric_cols].corr()['Attrition'].drop('Attrition')

# Sort by absolute value — strongest relationship first

correlation_sorted = correlation.abs().sort_values(ascending=False)

# Show top 20

print("Top 20 Correlations with Attrition:")
print(correlation_sorted.head(20).to_frame('correlation').to_string())

Top 20 Correlations with Attrition:
                                  correlation
OverTime                             0.211506
TotalWorkingYears                    0.154905
JobLevel                             0.149929
YearsInCurrentRole                   0.149268
MaritalStatus_Single                 0.145681
YearsWithCurrManager                 0.143163
Age                                  0.138441
JobRole_Sales Representative         0.136137
MonthlyIncome                        0.127148
StockOptionLevel                     0.111451
JobInvolvement                       0.110310
YearsAtCompany                       0.108443
EnvironmentSatisfaction              0.101059
JobSatisfaction                      0.099742
DistanceFromHome                     0.094977
JobRole_Research Director            0.093671
MaritalStatus_Married                0.088716
BusinessTravel_Travel_Frequently     0.077833
JobRole_Manager                      0.074466
EducationField_Technical Degree      0.06363

In [11]:
# Feature Selection

selected_features = correlation_sorted[
    correlation_sorted > 0.10
].index.tolist()

# Add JobSatisfaction based on domain knowledge

selected_features.append('JobSatisfaction')

print(f" Selected {len(selected_features)} features:")
for i, f in enumerate(selected_features, 1):
    print(f"   {i:2}. {f:<35} {correlation_sorted[f]:.4f}")

 Selected 14 features:
    1. OverTime                            0.2115
    2. TotalWorkingYears                   0.1549
    3. JobLevel                            0.1499
    4. YearsInCurrentRole                  0.1493
    5. MaritalStatus_Single                0.1457
    6. YearsWithCurrManager                0.1432
    7. Age                                 0.1384
    8. JobRole_Sales Representative        0.1361
    9. MonthlyIncome                       0.1271
   10. StockOptionLevel                    0.1115
   11. JobInvolvement                      0.1103
   12. YearsAtCompany                      0.1084
   13. EnvironmentSatisfaction             0.1011
   14. JobSatisfaction                     0.0997


In [12]:
from scipy.stats import chi2_contingency

# Creating categorical columns
#Domain Knowledge and Text Columns in original dataset leads to caregorical decision (Manual decision)

categorical_cols = ['Department', 'JobRole', 'MaritalStatus',
                    'BusinessTravel', 'EducationField', 'Gender'] 

print("Chi-Square Test Results:")
print(f"{'Column':<25} {'Chi2':>10} {'P-Value':>10} {'Significant':>12}")
print("-" * 60)

for col in categorical_cols:
    # Build contingency table
    contingency = pd.crosstab(df[col], df['Attrition'])
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    significant = "Yes" if p_value < 0.05 else "No"
    print(f"{col:<25} {chi2:>10.2f} {p_value:>10.4f} {significant:>12}")

Chi-Square Test Results:
Column                          Chi2    P-Value  Significant
------------------------------------------------------------
Department                      6.20     0.0451          Yes
JobRole                        65.70     0.0000          Yes
MaritalStatus                  32.80     0.0000          Yes
BusinessTravel                 11.22     0.0106          Yes
EducationField                 14.79     0.0113          Yes
Gender                          1.36     0.5064           No


In [13]:
# Remove Gender from df_model

cols_to_remove = [col for col in df_model.columns 
                  if 'Gender' in col]

print(f"Removing gender columns: {cols_to_remove}")
df_model = df_model.drop(columns=cols_to_remove)

# Final feature list

final_features = selected_features.copy()

# Rebuild X and y with validated features only

X = df_model[final_features]
y = df_model['Attrition']

print(f"\n Final validated features: {len(final_features)}")
print(f"   Shape of X: {X.shape}")
print(f"   Shape of y: {y.shape}")

Removing gender columns: ['Gender_Male', 'Gender_Male ']

 Final validated features: 14
   Shape of X: (1544, 14)
   Shape of y: (1544,)


In [14]:
# Split with validated features only
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train/Test split rebuilt!")
print(f"   Training set:  {X_train.shape[0]:,} rows")
print(f"   Testing set:   {X_test.shape[0]:,} rows")
print(f"   Features:      {X_train.shape[1]}")

Train/Test split rebuilt!
   Training set:  1,235 rows
   Testing set:   309 rows
   Features:      14


In [17]:
# Build model

model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight='balanced'
)

# Train model

model.fit(X_train_scaled, y_train)

# Predict on test data

y_pred = model.predict(X_test_scaled)

# Model accuracy

accuracy = accuracy_score(y_test, y_pred)
print(f" Model trained!")
print(f" Accuracy: {accuracy:.2%}")


 Model trained!
 Accuracy: 65.70%


In [18]:
# Cell 12 — Detailed Model Evaluation

print(" Classification Report:")
print(classification_report(y_test, y_pred, 
      target_names=['Stayed', 'Left']))

print("\n Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"                 Predicted")
print(f"                 Stayed  Left")
print(f"Actual  Stayed   {cm[0][0]:5}  {cm[0][1]:5}")
print(f"Actual  Left     {cm[1][0]:5}  {cm[1][1]:5}")

 Classification Report:
              precision    recall  f1-score   support

      Stayed       0.87      0.67      0.76       245
        Left       0.33      0.61      0.42        64

    accuracy                           0.66       309
   macro avg       0.60      0.64      0.59       309
weighted avg       0.76      0.66      0.69       309


 Confusion Matrix:
                 Predicted
                 Stayed  Left
Actual  Stayed     164     81
Actual  Left        25     39


In [19]:
from sklearn.ensemble import RandomForestClassifier

# Build Random Forest

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    max_depth=10
)

# Train

rf_model.fit(X_train_scaled, y_train)

# Predict

y_pred_rf = rf_model.predict(X_test_scaled)

# Accuracy

rf_accuracy = accuracy_score(y_test, y_pred_rf)
print(f" Random Forest trained!")
print(f"   Accuracy: {rf_accuracy:.2%}")

# Detailed report

print("\n Classification Report:")
print(classification_report(y_test, y_pred_rf,
      target_names=['Stayed', 'Left']))

# Confusion Matrix

cm_rf = confusion_matrix(y_test, y_pred_rf)
print(" Confusion Matrix:")
print(f"                 Predicted")
print(f"                 Stayed  Left")
print(f"Actual  Stayed   {cm_rf[0][0]:5}  {cm_rf[0][1]:5}")
print(f"Actual  Left     {cm_rf[1][0]:5}  {cm_rf[1][1]:5}")

 Random Forest trained!
   Accuracy: 81.23%

 Classification Report:
              precision    recall  f1-score   support

      Stayed       0.84      0.94      0.89       245
        Left       0.59      0.31      0.41        64

    accuracy                           0.81       309
   macro avg       0.71      0.63      0.65       309
weighted avg       0.79      0.81      0.79       309

 Confusion Matrix:
                 Predicted
                 Stayed  Left
Actual  Stayed     231     14
Actual  Left        44     20


In [20]:
from xgboost import XGBClassifier

# Build XGBoost model

xgb_model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    scale_pos_weight=4,
    max_depth=6,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Train

xgb_model.fit(X_train_scaled, y_train)

# Predict

y_pred_xgb = xgb_model.predict(X_test_scaled)

# Accuracy

xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
print(f"✅ XGBoost trained!")
print(f"   Accuracy: {xgb_accuracy:.2%}")

# Detailed report

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred_xgb,
      target_names=['Stayed', 'Left']))

# Confusion Matrix

cm_xgb = confusion_matrix(y_test, y_pred_xgb)
print("📊 Confusion Matrix:")
print(f"                 Predicted")
print(f"                 Stayed  Left")
print(f"Actual  Stayed   {cm_xgb[0][0]:5}  {cm_xgb[0][1]:5}")
print(f"Actual  Left     {cm_xgb[1][0]:5}  {cm_xgb[1][1]:5}")


✅ XGBoost trained!
   Accuracy: 77.35%

📊 Classification Report:
              precision    recall  f1-score   support

      Stayed       0.87      0.84      0.86       245
        Left       0.46      0.50      0.48        64

    accuracy                           0.77       309
   macro avg       0.66      0.67      0.67       309
weighted avg       0.78      0.77      0.78       309

📊 Confusion Matrix:
                 Predicted
                 Stayed  Left
Actual  Stayed     207     38
Actual  Left        32     32


/Users/akshayraghunath/opt/anaconda3/lib/python3.9/site-packages/xgboost/sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")


In [21]:
# Compare all 3 models

comparison = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Random Forest',
        'XGBoost'
    ],
    'Accuracy': [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],
    'Leaver Recall': [
        confusion_matrix(y_test, y_pred)[1][1] / 64,
        confusion_matrix(y_test, y_pred_rf)[1][1] / 64,
        confusion_matrix(y_test, y_pred_xgb)[1][1] / 64
    ],
    'Leavers Caught': [
        confusion_matrix(y_test, y_pred)[1][1],
        confusion_matrix(y_test, y_pred_rf)[1][1],
        confusion_matrix(y_test, y_pred_xgb)[1][1]
    ],
    'False Alarms': [
        confusion_matrix(y_test, y_pred)[0][1],
        confusion_matrix(y_test, y_pred_rf)[0][1],
        confusion_matrix(y_test, y_pred_xgb)[0][1]
    ]
})

comparison['Accuracy']      = comparison['Accuracy'].apply(lambda x: f"{x:.1%}")
comparison['Leaver Recall'] = comparison['Leaver Recall'].apply(lambda x: f"{x:.1%}")

print("Model Comparison:")
print(comparison.to_string(index=False))

Model Comparison:
              Model Accuracy Leaver Recall  Leavers Caught  False Alarms
Logistic Regression    65.7%         60.9%              39            81
      Random Forest    81.2%         31.2%              20            14
            XGBoost    77.3%         50.0%              32            38


In [22]:
# Risk Score - Get probability of leaving for ALL employees

X_all_scaled = scaler.transform(X[final_features])
risk_scores   = model.predict_proba(X_all_scaled)[:, 1]

# Add to original dataframe

df_scored = df.copy()
df_scored['risk_score']   = risk_scores.round(4)
df_scored['risk_score_%'] = (risk_scores * 100).round(2)

# Assign risk tiers

df_scored['risk_tier'] = pd.cut(
    risk_scores,
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low', 'Medium', 'High']
)

# Summary

print("Risk Tier Distribution:")
print(df_scored['risk_tier'].value_counts())
print(f"\n High Risk:   {(df_scored['risk_tier']=='High').sum():,}")
print(f" Medium Risk: {(df_scored['risk_tier']=='Medium').sum():,}")
print(f" Low Risk:    {(df_scored['risk_tier']=='Low').sum():,}")

Risk Tier Distribution:
Medium    753
Low       428
High      363
Name: risk_tier, dtype: int64

 High Risk:   363
 Medium Risk: 753
 Low Risk:    428


In [23]:
# Cell 18 — Financial Impact Calculation

avg_monthly_salary   = df_scored['MonthlyIncome'].mean()
replacement_cost     = avg_monthly_salary * 6

high_risk_count      = 363
total_risk_exposure  = high_risk_count * replacement_cost

print(" Financial Impact Report:")
print(f"   Average Monthly Salary:    ${avg_monthly_salary:,.0f}")
print(f"   Replacement Cost Each:     ${replacement_cost:,.0f}")
print(f"   High Risk Employees:       {high_risk_count:,}")
print(f"   Total Risk Exposure:       ${total_risk_exposure:,.0f}")
print(f"   Annual Risk Exposure:      ${total_risk_exposure:,.0f}")

 Financial Impact Report:
   Average Monthly Salary:    $6,404
   Replacement Cost Each:     $38,425
   High Risk Employees:       363
   Total Risk Exposure:       $13,948,095
   Annual Risk Exposure:      $13,948,095


In [24]:
# Cell 19 — Export All Results

from sqlalchemy import create_engine

# Export scored employees to CSV

df_scored.to_csv(exports_path + "employee_risk_scores.csv", index=False)
print("✅ Exported: employee_risk_scores.csv")

# Export to PostgreSQL
df_scored.to_sql('employee_risk_scores', engine, 
                 if_exists='replace', index=False)
print("✅ Loaded to PostgreSQL: employee_risk_scores")

# Export model comparison
model_comparison = pd.DataFrame({
    'Model':          ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy':       ['65.7%', '81.2%', '77.3%'],
    'Leaver_Recall':  ['60.9%', '31.2%', '50.0%'],
    'Leavers_Caught': [39, 20, 32],
    'False_Alarms':   [81, 14, 38],
    'Selected':       ['Yes', 'No', 'NO']
})

model_comparison.to_csv(exports_path + "model_comparison.csv", index=False)
print(" Exported: model_comparison.csv")

print(f"\n All files saved to: {exports_path}")

✅ Exported: employee_risk_scores.csv
✅ Loaded to PostgreSQL: employee_risk_scores
 Exported: model_comparison.csv

 All files saved to: /Users/akshayraghunath/Documents/ibmhr/outputs/exports/
